In [1]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!git clone https://github.com/yugisu/SatDiFuser.git
!cd SatDiFuser && git checkout research

import sys

sys.path.append("./SatDiFuser")

fatal: destination path 'SatDiFuser' already exists and is not an empty directory.
Already on 'research'
Your branch is up to date with 'origin/research'.


In [ ]:
!cd SatDiFuser && uv pip install -r pyproject.toml

# Addressing `diffusionsat` and `diffusers` import issues
!uv pip install gdown lightning
!uv pip install "jax[cuda12_local]==0.4.23" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
!uv pip uninstall flax

# RESTART THE NOTEBOOK AFTER INSTALL

Using Python 3.12.12 environment at: /usr
Resolved 89 packages in 39ms                                         
Audited 89 packages in 1ms
Using Python 3.12.12 environment at: /usr
Audited 1 package in 91ms
Using Python 3.12.12 environment at: /usr


In [4]:
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn.functional as F
from PIL import Image
from rasterio.enums import Resampling
from torch import nn
from torchvision import transforms
from tqdm.notebook import tqdm

warnings.filterwarnings(
  "ignore", category=SyntaxWarning, message=".*invalid escape sequence.*"
)

DEVICE = torch.device("cuda")

## Data


In [5]:
VISLOC_ROOT = Path("./data/visloc")

FLIGHT_ID = "03"
SAT_SCALE = 0.2
CHUNK_SIZE = 256
CHUNK_STRIDE = 128


# Download VisLoc dataset if needed.
import gdown

if not VISLOC_ROOT.exists():
  zip_path = "data/visloc.zip"
  Path("data").mkdir(exist_ok=True)
  gdown.download(id="16tY7tPZiNIoyAhknvyXnp0jAfccIcHtL", output=zip_path)
  import zipfile

  with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall("data/visloc")
  # Fix renamed CSV
  old = VISLOC_ROOT / "satellite_ coordinates_range.csv"
  new = VISLOC_ROOT / "satellite_coordinates_range.csv"
  if old.exists() and not new.exists():
    old.rename(new)
else:
  print(f'"{VISLOC_ROOT}" already exists. Skipping download.')

Downloading...
From (original): https://drive.google.com/uc?id=16tY7tPZiNIoyAhknvyXnp0jAfccIcHtL
From (redirected): https://drive.google.com/uc?id=16tY7tPZiNIoyAhknvyXnp0jAfccIcHtL&confirm=t&uuid=c3defa6a-e732-4edf-b74c-8e9851fc5ded
To: /content/data/visloc.zip
100%|██████████| 2.19G/2.19G [00:54<00:00, 40.2MB/s]


In [6]:
# --- Satellite chunk helpers (adapted from 0-explore-visloc.ipynb) ---


def load_satellite_image(data_path: Path, flight_id: str, scale_factor: float = 0.1):
  """Load and downscale satellite image.

  Returns:
    sat_img: (H, W, C) uint8 numpy array
    bounds:  (lat_min, lon_min, lat_max, lon_max)
    dims:    dict with height, width, scale_factor
  """
  sat_path = data_path / flight_id / f"satellite{flight_id}.tif"
  with rasterio.open(sat_path) as src:
    h = int(src.height * scale_factor)
    w = int(src.width * scale_factor)
    data = src.read(out_shape=(src.count, h, w), resampling=Resampling.lanczos)
    sat_img = (
      np.transpose(data[:3], (1, 2, 0)).astype(np.uint8)
      if src.count >= 3
      else np.transpose(data[0:1], (1, 2, 0)).astype(np.uint8)
    )

  sat_df = pd.read_csv(data_path / "satellite_coordinates_range.csv")
  sat_df.columns = ["mapname", "LT_lat", "LT_lon", "RB_lat", "RB_lon", "region"]
  row = sat_df[sat_df["mapname"] == f"{flight_id}.tif"].iloc[0]

  bounds = (row["RB_lat"], row["LT_lon"], row["LT_lat"], row["RB_lon"])
  dims = {"height": h, "width": w, "scale_factor": scale_factor}
  return sat_img, bounds, dims


def create_chunks(
  sat_img: np.ndarray,
  bounds: tuple,
  dims: dict,
  chunk_size: int = 256,
  stride: int | None = None,
) -> list[dict]:
  """Split satellite image into overlapping chunks with GPS coordinates."""
  stride = stride or chunk_size
  h, w = dims["height"], dims["width"]
  lat_min, lon_min, lat_max, lon_max = bounds

  def pixel_to_gps(x, y):
    lat = lat_max - (y / h) * (lat_max - lat_min)
    lon = lon_min + (x / w) * (lon_max - lon_min)
    return lat, lon

  chunks = []
  for y in range(0, h - chunk_size, stride):
    for x in range(0, w - chunk_size, stride):
      chunk = sat_img[y : y + chunk_size, x : x + chunk_size]
      cx, cy = x + chunk_size // 2, y + chunk_size // 2
      lat, lon = pixel_to_gps(cx, cy)
      chunk_lat_min, chunk_lon_min = pixel_to_gps(x, y + chunk_size)
      chunk_lat_max, chunk_lon_max = pixel_to_gps(x + chunk_size, y)
      chunks.append(
        {
          "rgb": chunk,
          "lat": lat,
          "lon": lon,
          "bbox_gps": (chunk_lat_min, chunk_lon_min, chunk_lat_max, chunk_lon_max),
        }
      )

  print(
    f"Created {len(chunks)} chunks from {w}x{h} image (scale={dims['scale_factor']})"
  )
  return chunks


# --- Datasets ---


class UAVDataset(torch.utils.data.Dataset):
  """VisLoc UAV drone image dataset."""

  def __init__(
    self,
    root: Path,
    flight_id: str,
    transform: transforms.Compose | None = None,
  ) -> None:
    self.drone_dir = root / flight_id / "drone"
    self.transform = transform
    csv_path = root / flight_id / f"{flight_id}.csv"
    df = pd.read_csv(csv_path)
    self.records = df[["filename", "lat", "lon"]].reset_index(drop=True)

  def __len__(self) -> int:
    return len(self.records)

  def __getitem__(self, index: int):
    row = self.records.iloc[index]
    img = Image.open(self.drone_dir / row["filename"]).convert("RGB")
    if self.transform is not None:
      img = self.transform(img)
    return img, float(row["lat"]), float(row["lon"])


class SatChunkDataset(torch.utils.data.Dataset):
  """Satellite image chunks as a PyTorch dataset."""

  def __init__(
    self,
    chunks: list[dict],
    transform: transforms.Compose | None = None,
  ) -> None:
    self.chunks = chunks
    self.transform = transform

  def __len__(self) -> int:
    return len(self.chunks)

  def __getitem__(self, index: int):
    chunk = self.chunks[index]
    img = Image.fromarray(chunk["rgb"])
    if self.transform is not None:
      img = self.transform(img)
    return img, float(chunk["lat"]), float(chunk["lon"])

## Definitions


In [7]:
# Defaults for LDM feature extraction.


from archs.ldm_extractor import LDMExtractor
from omegaconf import OmegaConf


def ldm_extractor_cfg(
  img_size=256,
  batch_size=8,
  save_timesteps=[48, 46, 42],
  layer_idxs={"down_blocks": {"attn1": "all"}},
  num_timesteps=50,
  diffusion_mode="inversion",
  prompt="A satellite image",
  negative_prompt="",
  resize_outputs=-1,
  max_i=None,
  min_i=None,
):
  cfg: dict = {
    "num_timesteps": num_timesteps,
    "save_timesteps": save_timesteps,
    "batch_size": batch_size,
    "img_size": img_size,
    "resize_outputs": resize_outputs,
    "diffusion_mode": diffusion_mode,
    "layer_idxs": layer_idxs,
    "prompt": prompt,
    "negative_prompt": negative_prompt,
  }
  if max_i is not None:
    cfg["max_i"] = max_i
  if min_i is not None:
    cfg["min_i"] = min_i

  return OmegaConf.create(cfg)


def fix_seed(seed):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  torch.cuda.manual_seed(seed)
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False


In [8]:
# Evaluation.
#
# For u2s retrieval, each UAV image may match multiple satellite chunks (overlapping tiles).
# ground_truth is a list of sets of valid chunk indices.


def build_ground_truth(
  uav_lats: np.ndarray,
  uav_lons: np.ndarray,
  chunks: list[dict],
) -> list[set]:
  """For each UAV image, find all satellite chunk indices whose GPS bbox contains it.
  Falls back to the nearest chunk if no bbox contains the point."""
  ground_truth = []
  for lat, lon in zip(uav_lats, uav_lons):
    matching = set()
    for i, chunk in enumerate(chunks):
      lat_min, lon_min, lat_max, lon_max = chunk["bbox_gps"]
      if lat_min <= lat <= lat_max and lon_min <= lon <= lon_max:
        matching.add(i)
    if not matching:
      dists = [(lat - c["lat"]) ** 2 + (lon - c["lon"]) ** 2 for c in chunks]
      matching = {int(np.argmin(dists))}
    ground_truth.append(matching)
  return ground_truth


def recall_at_k(preds: np.ndarray, ground_truth: list[set], k: int) -> float:
  """Recall@k with multi-match ground truth."""
  hits = sum(any(p in gt for p in preds[i, :k]) for i, gt in enumerate(ground_truth))
  return hits / len(ground_truth)


def calculate_metrics(preds: np.ndarray, ground_truth: list[set]) -> dict:
  return {
    "Recall@1": recall_at_k(preds, ground_truth, k=1),
    "Recall@5": recall_at_k(preds, ground_truth, k=5),
    "Recall@10": recall_at_k(preds, ground_truth, k=10),
  }

## Main


In [ ]:
# Frozen backbone initialization.


DIFFUSIONSAT_CHKPTS = Path("/content/drive/MyDrive/diffusionsat-checkpoints")
chkpt256 = DIFFUSIONSAT_CHKPTS / "finetune_sd21_256_sn-satlas-fmow_snr5_md7norm_bs64"
chkpt512 = DIFFUSIONSAT_CHKPTS / "finetune_sd21_sn-satlas-fmow_snr5_md7norm_bs64"

chkpt = chkpt256
revision = None


from diffusers import AutoencoderKL
from diffusionsat import DiffusionSatPipeline, SatUNet

unet = SatUNet.from_pretrained(
  chkpt,
  subfolder="checkpoint-150000/unet",
  revision=revision,
  num_metadata=0,
  use_metadata=False,
  low_cpu_mem_usage=False,
)
unet.requires_grad_(False)
unet.to(DEVICE)

pipe = DiffusionSatPipeline.from_pretrained(chkpt, unet=unet, low_cpu_mem_usage=False)
pipe.to(DEVICE)

vae = AutoencoderKL.from_pretrained(chkpt, subfolder="vae", revision=revision)
vae.requires_grad_(False)
vae.to(DEVICE)

ldm_cfg = ldm_extractor_cfg()
ldm_extractor = LDMExtractor(ldm_cfg, pipe)

/usr/local/lib/python3.12/dist-packages/diffusers/models/cross_attention.py:30: FutureWarning: Importing from cross_attention is deprecated. Please import from diffusers.models.attention_processor instead.
  deprecate(
Some weights of the model checkpoint at /content/drive/MyDrive/diffusionsat-checkpoints/finetune_sd21_256_sn-satlas-fmow_snr5_md7norm_bs64 were not used when initializing SatUNet: ['metadata_embedding.1.linear_2.bias', 'metadata_embedding.0.linear_1.bias', 'metadata_embedding.6.linear_2.weight', 'metadata_embedding.4.linear_1.weight', 'metadata_embedding.1.linear_1.weight', 'metadata_embedding.4.linear_1.bias', 'metadata_embedding.2.linear_2.bias', 'metadata_embedding.2.linear_1.weight', 'metadata_embedding.5.linear_2.weight', 'metadata_embedding.6.linear_2.bias', 'metadata_embedding.1.linear_2.weight', 'metadata_embedding.0.linear_1.weight', 'metadata_embedding.4.linear_2.bias', 'metadata_embedding.0.linear_2.bias', 'metadata_embedding.0.linear_2.weight', 'metadata_embe

### Train the fuser-based model


In [10]:
cfg = {
  # Training
  "lr": 1e-3,
  "min_lr": 1e-5,
  "warmup_epochs": 1,
  "max_train_epochs": 3,
  "op_weight_decay": 1e-4,
  "batch_size": 8,
  "val_batch_size": 8,
  "num_workers": 4,
  "eval_every_n_steps": None,
  "early_stopping_patience": 10,
  "seed": 42,
}

In [11]:
import math

from archs.embedders import FuserEmbedder
from archs.losses import SpatialLoss
from archs.retrievers import FAISSRetriever

g = torch.Generator(device=DEVICE).manual_seed(cfg["seed"])
fix_seed(cfg["seed"])


transform = transforms.Compose(
  [
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
  ]
)

sat_img, bounds, dims = load_satellite_image(
  VISLOC_ROOT, FLIGHT_ID, scale_factor=SAT_SCALE
)
chunks = create_chunks(
  sat_img, bounds, dims, chunk_size=CHUNK_SIZE, stride=CHUNK_STRIDE
)

sat_dataset = SatChunkDataset(chunks, transform=transform)
sat_loader = torch.utils.data.DataLoader(
  sat_dataset,
  batch_size=cfg["batch_size"],
  shuffle=True,
  num_workers=cfg["num_workers"],
)

uav_dataset = UAVDataset(VISLOC_ROOT, FLIGHT_ID, transform=transform)
uav_loader = torch.utils.data.DataLoader(
  uav_dataset,
  batch_size=cfg["val_batch_size"],
  shuffle=False,
  num_workers=cfg["num_workers"],
)

ground_truth = build_ground_truth(
  np.array([chunks[i]["lat"] for i in range(len(chunks))]),
  np.array([chunks[i]["lon"] for i in range(len(chunks))]),
  chunks,
)

embedder = FuserEmbedder(
  save_timesteps=ldm_cfg.save_timesteps,
  feature_dims=ldm_extractor.collected_dims,
).to(DEVICE)

# alpha=1.0: global VICReg only (no local spatial matching).
# The local VICRegL term is O(num_matches * C^2) where C=projection_dim*num_scales (~1152),
# which exceeds available memory even with small num_matches.
criterion = SpatialLoss(alpha=1.0)

# --- Augmentation applied at image level before VAE encoding ---


def augment_images(imgs: torch.Tensor) -> torch.Tensor:
  """Random spatial flip augmentation on a batch of images."""

  # TODO: UAV-like augmentations like in CAEVL.

  if torch.rand(1).item() > 0.5:
    imgs = imgs.flip(-1)  # horizontal flip
  if torch.rand(1).item() > 0.5:
    imgs = imgs.flip(-2)  # vertical flip
  return imgs


def encode(imgs: torch.Tensor) -> torch.Tensor:
  return vae.encode(imgs).latent_dist.sample(generator=g) * 0.18215


# --- Optimizer and LR schedule ---

optimizer = torch.optim.AdamW(
  embedder.parameters(),
  lr=cfg["lr"],
  weight_decay=cfg["op_weight_decay"],
)

steps_per_epoch = len(sat_loader)
total_steps = cfg["max_train_epochs"] * steps_per_epoch
warmup_steps = cfg["warmup_epochs"] * steps_per_epoch


def lr_lambda(step):
  if step < warmup_steps:
    return step / max(1, warmup_steps)
  t = (step - warmup_steps) / max(1, total_steps - warmup_steps)
  return max(cfg["min_lr"] / cfg["lr"], 0.5 * (1.0 + math.cos(math.pi * t)))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# --- Evaluation helper ---


@torch.no_grad()
def extract_all_embeddings(data_loader, desc="Embeddings"):
  embedder.eval()
  emb_list = []
  for imgs, _, _ in tqdm(data_loader, desc=desc, leave=False):
    imgs = imgs.to(DEVICE)
    latents = encode(imgs)
    feats, _ = ldm_extractor.forward(latents)
    _, pooled = embedder.forward_unpooled(feats)
    emb_list.append(F.normalize(pooled, p=2, dim=1).cpu())
  return torch.cat(emb_list)


def evaluate():
  gallery_embs = extract_all_embeddings(sat_loader, "Gallery")
  query_embs = extract_all_embeddings(uav_loader, "Query")
  retriever = FAISSRetriever(gallery_embs)
  _, preds = retriever.search(query_embs, k=10)
  return calculate_metrics(preds, ground_truth)


# --- Training loop ---

best_recall1 = -1.0
best_state = None
patience_counter = 0
global_step = 0

for epoch in range(cfg["max_train_epochs"]):
  embedder.train()
  epoch_loss = 0.0

  tbar = tqdm(sat_loader, desc=f"Epoch {epoch + 1}", leave=False)
  for imgs, _, _ in tbar:
    imgs = imgs.to(DEVICE)

    # Two augmented views encoded through frozen VAE
    with torch.no_grad():
      view1 = augment_images(imgs)
      view2 = augment_images(imgs)
      latents1 = encode(view1)
      latents2 = encode(view2)
      feats1, _ = ldm_extractor.forward(latents1)
      feats2, _ = ldm_extractor.forward(latents2)

    optimizer.zero_grad()
    spatial1, pooled1 = embedder.forward_unpooled(feats1)
    spatial2, pooled2 = embedder.forward_unpooled(feats2)
    loss = criterion(spatial1, pooled1, spatial2, pooled2)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(embedder.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

    epoch_loss += loss.item()
    global_step += 1
    tbar.set_postfix(loss=f"{loss.item():.4f}")

    if cfg["eval_every_n_steps"] and global_step % cfg["eval_every_n_steps"] == 0:
      metrics = evaluate()
      print(
        f"  step={global_step}  "
        + "  ".join(f"{k}={v:.4f}" for k, v in metrics.items())
      )
      embedder.train()

  avg_loss = epoch_loss / steps_per_epoch
  lr_now = scheduler.get_last_lr()[0]
  metrics = evaluate()
  r1 = metrics["Recall@1"]
  print(
    f"Epoch {epoch + 1:2d}/{cfg['max_train_epochs']}  "
    f"loss={avg_loss:.4f}  lr={lr_now:.2e}  "
    + "  ".join(f"{k}={v:.4f}" for k, v in metrics.items())
  )

  if r1 > best_recall1:
    best_recall1 = r1
    patience_counter = 0
    best_state = {k: v.clone() for k, v in embedder.state_dict().items()}
  else:
    patience_counter += 1
    if patience_counter >= cfg["early_stopping_patience"]:
      print(
        f"Early stopping at epoch {epoch + 1} (no improvement for {patience_counter} epochs)"
      )
      break

if best_state is not None:
  embedder.load_state_dict(best_state)
print(f"\nBest Recall@1: {best_recall1:.4f}")


Created 1908 chunks from 7018x4861 image (scale=0.2)


Epoch 1:   0%|          | 0/239 [00:00<?, ?it/s]

Gallery:   0%|          | 0/239 [00:00<?, ?it/s]

Query:   0%|          | 0/96 [00:00<?, ?it/s]

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <cell line: 0>:171                                                                            │
│                                                                                                  │
│   168                                                                                            │
│   169   avg_loss = epoch_loss / steps_per_epoch                                                  │
│   170   lr_now = scheduler.get_last_lr()[0]                                                      │
│ ❱ 171   metrics = evaluate()                                                                     │
│   172   r1 = metrics["Recall@1"]                                                                 │
│   173   print(                                                                                   │
│   174 │   f"Epoch {epoch + 1:2d}/{cfg['max_train_epochs']}  "                                    │
│                                                                                                  │
│ in evaluate:121                                                                                  │
│                                                                                                  │
│   118   query_embs = extract_all_embeddings(uav_loader, "Query")                                 │
│   119   retriever = FAISSRetriever(gallery_embs)                                                 │
│   120   _, preds = retriever.search(query_embs, k=10)                                            │
│ ❱ 121   return calculate_metrics(preds, ground_truth)                                            │
│   122                                                                                            │
│   123                                                                                            │
│   124 # --- Training loop ---                                                                    │
│                                                                                                  │
│ in calculate_metrics:36                                                                          │
│                                                                                                  │
│   33                                                                                             │
│   34 def calculate_metrics(preds: np.ndarray, ground_truth: list[set]) -> dict:                  │
│   35   return {                                                                                  │
│ ❱ 36 │   "Recall@1": recall_at_k(preds, ground_truth, k=1),                                      │
│   37 │   "Recall@5": recall_at_k(preds, ground_truth, k=5),                                      │
│   38 │   "Recall@10": recall_at_k(preds, ground_truth, k=10),                                    │
│   39   }                                                                                         │
│                                                                                                  │
│ in recall_at_k:30                                                                                │
│                                                                                                  │
│   27                                                                                             │
│   28 def recall_at_k(preds: np.ndarray, ground_truth: list[set], k: int) -> float:               │
│   29   """Recall@k with multi-match ground truth."""                                             │
│ ❱ 30   hits = sum(any(p in gt for p in preds[i, :k]) for i, gt in enumerate(ground_truth))       │
│   31   return hits / len(ground_truth)                                                           │
│   32                                                                                             │
│   33                                                       

### Evaluate the fuser-based model


In [ ]:
# TODO: Build a gallery/database of embeddings of satellite chunks

In [ ]:
# TODO: Evaluate the trained model on uav-to-sat retrieval metrics
